# GSAD (NeurIPS 2023, diffusion) — test on LOL-v1 / LOL-v2-real / LOL-v2-syn (Kaggle)

**Settings:** Accelerator = GPU (T4 or P100), Internet = ON. Then **Save Version → Save & Run All**. ~20–40 min.

**Why a patched test script?** GSAD's own `test.py` ALWAYS rescales every output so its average brightness equals
the GROUND TRUTH's (lines "mean_gray_gt / mean_gray_out"). A real camera has no ground truth, so this uses
information from the answer and inflates PSNR. We save BOTH versions:
* `noadjust/` — the network's real output  → **our main numbers** (comparable with every other method)
* `gtmean/`  — the authors' brightness-corrected output → only to check we reproduce the paper's numbers

We also time the diffusion sampling per image (GPU-synchronised, first image skipped as warm-up).

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code + dependencies. We do NOT install their pinned old versions (opencv 4.5, scikit-image 0.18 ...):
#    Kaggle's newer versions work, and downgrading would break Kaggle's PyTorch.
%cd /kaggle/working
!git clone -q https://github.com/jinnh/GSAD.git
%cd /kaggle/working/GSAD
# kmeans_pytorch + tensorboardX: imported by GSAD's diffusion/logger code (verified by importing every
# GSAD module on the Mac); the rest of its imports (cv2, skimage, sklearn, tensorboard, ...) Kaggle already has.
!pip install -q gdown lpips natsort kmeans_pytorch tensorboardX

In [ ]:
# 3. Data (same Google Drive zips as Retinexformer's README) + the folder names GSAD expects:
#    LOLv1: dataset/LOLv1/eval15/{low,high}   (ours: LOLv1/Test/{input,target})
#    LOLv2: dataset/LOLv2/{Real_captured,Synthetic}/Test/{Low,Normal}  (same as ours)
import os, subprocess
def gdl(fid, out):
    for _ in range(3):                                   # gdown is sometimes flaky -> retry
        if subprocess.call(f"gdown -q {fid} -O {out}", shell=True) == 0:
            return
    raise RuntimeError("download failed: " + out)
gdl("1L-kqSQyrmMueBh_ziWoPFhfsAh50h20H", "/kaggle/tmp_lolv1.zip")
gdl("1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U", "/kaggle/tmp_lolv2.zip")
!mkdir -p /kaggle/lol && unzip -q -o /kaggle/tmp_lolv1.zip -d /kaggle/lol && unzip -q -o /kaggle/tmp_lolv2.zip -d /kaggle/lol
!mkdir -p dataset/LOLv1/eval15 dataset/LOLv1/our485
!ln -sfn /kaggle/lol/LOLv1/Test/input dataset/LOLv1/eval15/low && ln -sfn /kaggle/lol/LOLv1/Test/target dataset/LOLv1/eval15/high
!ln -sfn /kaggle/lol/LOLv1/Train/input dataset/LOLv1/our485/low && ln -sfn /kaggle/lol/LOLv1/Train/target dataset/LOLv1/our485/high
!ln -sfn /kaggle/lol/LOLv2 dataset/LOLv2
!echo "v1 test: $(ls dataset/LOLv1/eval15/low | wc -l) (15)  v2-real: $(ls dataset/LOLv2/Real_captured/Test/Low | wc -l) (100)  v2-syn: $(ls dataset/LOLv2/Synthetic/Test/Low | wc -l) (100)"

In [ ]:
# 4. Official pretrained weights (README "pretrained model" Google Drive folder)
!mkdir -p checkpoints
for fid, name in [("1Qx4y04tPsKwF_WJypY9HZc7K4BVm4f8H", "lolv1_gen.pth"),
                  ("1ZCRRpYBbZ6PUC4KTpIj7WN421ZN1BGd-", "lolv2_real_gen.pth"),
                  ("1hDWvOfNkjeKPJfW2kVzZS2WlLDipI3Tr", "lolv2_syn_gen.pth")]:
    gdl(fid, f"checkpoints/{name}")
!ls -la checkpoints

In [ ]:
# 5. Make test_btp.py = their test.py + 3 small additions (their file stays untouched).
#    Each replace() is checked with assert, so if the upstream code changes we get a clear error, not silent nonsense.
src = open("test.py").read()

def patch(old, new):
    global src
    assert src.count(old) == 1, f"patch anchor not found exactly once:\n{old}"
    src = src.replace(old, new)

# (a) where to save, and a list for timings
patch("    lpipss = []\n",
      "    lpipss = []\n"
      "    import time, json\n"
      "    BTP_OUT = os.environ['BTP_OUT']\n"
      "    os.makedirs(BTP_OUT + '/noadjust', exist_ok=True); os.makedirs(BTP_OUT + '/gtmean', exist_ok=True)\n"
      "    _times = []\n")

# (b) time ONLY the diffusion sampling. cuda.synchronize = wait until the GPU has really finished.
patch("        diffusion.test(continous=False)\n",
      "        torch.cuda.synchronize(); _t0 = time.perf_counter()\n"
      "        diffusion.test(continous=False)\n"
      "        torch.cuda.synchronize(); _times.append(time.perf_counter() - _t0)\n"
      "        _name = val_data['LQ_path'][0]\n")

# (c) save the output BEFORE the ground-truth brightness trick ...
patch("        # Similar to LLFlow, we follow a similar way of 'Kind' to finetune the overall brightness \n",
      "        util.save_img(normal_img, f'{BTP_OUT}/noadjust/{_name}.png')   # BTP: real network output\n"
      "        # Similar to LLFlow, we follow a similar way of 'Kind' to finetune the overall brightness \n")
# ... and AFTER it (authors' protocol)
patch("        normal_img = (normal_img_adjust * 255).astype(np.uint8)\n",
      "        normal_img = (normal_img_adjust * 255).astype(np.uint8)\n"
      "        util.save_img(normal_img, f'{BTP_OUT}/gtmean/{_name}.png')     # BTP: GT-brightness-adjusted\n")

# (d) timing summary at the end (skip image 1 = warm-up)
patch("    logger_val.info('# Validation # avgPSNR",
      "    _t = _times[1:] if len(_times) > 1 else _times\n"
      "    json.dump({'n_images': len(_times), 'mean_s_per_image': float(np.mean(_t)),\n"
      "               'gpu': torch.cuda.get_device_name(0)}, open(BTP_OUT + '/timing.json', 'w'))\n"
      "    print(f'BTP timing: {np.mean(_t):.3f} s/image over {len(_t)} images on {torch.cuda.get_device_name(0)}')\n"
      "    logger_val.info('# Validation # avgPSNR")
open("test_btp.py", "w").write(src)
print("test_btp.py written")

In [ ]:
# 6. Run all three. PSNR printed at the end of each = AUTHORS' protocol (GT-mean) -> compare with the paper
#    (GSAD paper Table 1 — verify there; roughly LOLv1 27.8, LOLv2-real 28.8, LOLv2-syn 28.7 dB).
#    Our real numbers come from noadjust/ later.
runs = [("LOLv1",      "config/lolv1.yml",      "config/lolv1_test.json"),
        ("LOLv2-real", "config/lolv2_real.yml", "config/lolv2_real_test.json"),
        ("LOLv2-syn",  "config/lolv2_syn.yml",  "config/lolv2_syn_test.json")]
for ds, yml, js in runs:
    out = f"/kaggle/working/gsad/{ds}"
    print(f"\n========== {ds} ==========")
    os.makedirs(out, exist_ok=True)
    log = f"{out}/run.log"                      # full output kept, so an error is never hidden
    !BTP_OUT={out} python test_btp.py --dataset {yml} --config {js} > {log} 2>&1
    !grep -E "BTP timing|avgPSNR" {log}
    n = len(os.listdir(out + "/noadjust")) if os.path.isdir(out + "/noadjust") else 0
    if n == 0:
        !tail -n 25 {log}
        raise RuntimeError(f"{ds}: no images produced - see the 25 log lines above")
    print("saved:", n, "images (noadjust)")

In [ ]:
# 7. Zip for download: /kaggle/working/gsad.zip  (Output tab -> download)
!cd /kaggle/working && zip -q -r gsad.zip gsad && ls -la gsad.zip && cat gsad/*/timing.json